In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

In [2]:
np.random.seed(42)
n_samples = 1000

data = pd.DataFrame({
    'experience_years': np.random.randint(0, 20, n_samples),
    'education_level': np.random.choice([0, 1, 2], n_samples),  # 0=HS, 1=Bachelor, 2=Master
    'test_score': np.random.randint(60, 100, n_samples),
    'gender': np.random.choice(['M', 'F'], n_samples),
    'age': np.random.randint(22, 65, n_samples)
})

data.head()

,experience_years,education_level,test_score,gender,age
0,6,2,83,M,42
1,19,1,66,M,26
2,14,2,84,F,27
3,10,1,80,F,59
4,7,2,74,F,43


In [3]:
data['hired'] = (
    (data['experience_years'] > 5).astype(int) +
    (data['education_level'] >= 1).astype(int) +
    (data['test_score'] > 80).astype(int) +
    (data['gender'] == 'M').astype(int) * 0.3
) > 2

data.head()

,experience_years,education_level,test_score,gender,age,hired
0,6,2,83,M,42,True
1,19,1,66,M,26,True
2,14,2,84,F,27,True
3,10,1,80,F,59,False
4,7,2,74,F,43,False


In [4]:
X = data[['experience_years', 'education_level', 'test_score', 'age']]
y = data['hired']

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [7]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [8]:
predictions = model.predict(X_test)

data_test = data.iloc[X_test.index].copy()
data_test['prediction'] = predictions

data_test.head()

,experience_years,education_level,test_score,gender,age,hired,prediction
521,12,2,81,M,45,True,True
737,12,2,61,F,44,False,True
740,6,2,75,F,42,False,False
660,6,1,79,F,30,False,True
411,19,0,91,M,50,True,False


In [9]:
accuracy = (predictions == y_test).mean()
print(f"Overall Accuracy: {accuracy:.2%}")

Overall Accuracy: 77.00%


In [10]:
for gender in ['M', 'F']:
    gender_data = data_test[data_test['gender'] == gender]
    gender_accuracy = (gender_data['prediction'] == gender_data['hired']).mean()
    print(f"{gender} Accuracy: {gender_accuracy:.2%}")

M Accuracy: 75.97%
F Accuracy: 78.08%


In [11]:
for gender in ['M', 'F']:
    gender_data = data_test[data_test['gender'] == gender]
    positive_rate = gender_data['prediction'].mean()
    print(f"{gender} Positive Rate: {positive_rate:.2%}")

M Positive Rate: 42.21%
F Positive Rate: 41.10%


In [12]:
for gender in ['M', 'F']:
    gender_data = data_test[data_test['gender'] == gender]

    fp = ((gender_data['prediction'] == 1) & (gender_data['hired'] == 0)).sum()
    tn = ((gender_data['prediction'] == 0) & (gender_data['hired'] == 0)).sum()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

    fn = ((gender_data['prediction'] == 0) & (gender_data['hired'] == 1)).sum()
    tp = ((gender_data['prediction'] == 1) & (gender_data['hired'] == 1)).sum()
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

    print(f"\n{gender} Group:")
    print(f"False Positive Rate: {fpr:.2%}")
    print(f"False Negative Rate: {fnr:.2%}")


M Group:
False Positive Rate: 0.00%
False Negative Rate: 36.27%

F Group:
False Positive Rate: 27.12%
False Negative Rate: 0.00%


# Bias Audit Summary

## Objective
The objective of this exercise was to evaluate a machine learning hiring model for fairness and potential bias across gender groups.

## Key Findings
The model achieved an overall accuracy of 77%, indicating moderate predictive performance.

Accuracy by gender was relatively balanced:
- Male Accuracy: 75.97%
- Female Accuracy: 78.08%

Positive prediction rates were also similar:
- Male Positive Rate: 42.21%
- Female Positive Rate: 41.10%

This suggests little evidence of demographic parity bias.

However, fairness concerns were identified when analyzing error rates.

### False Positive Rate
- Male: 0.00%
- Female: 27.12%

The model incorrectly predicted unqualified female candidates as hired more often than male candidates.

### False Negative Rate
- Male: 36.27%
- Female: 0.00%

The model incorrectly rejected a significant number of qualified male candidates while rejecting none of the qualified female candidates.

## Ethical Concerns
Although overall performance appears balanced, the model shows unequal error rates across gender groups.

This indicates potential bias in decision-making and raises fairness concerns, particularly regarding equal opportunity.

## Recommendations
1. Investigate the cause of unequal false positive and false negative rates.
2. Audit training data for hidden bias.
3. Apply fairness-aware machine learning techniques.
4. Regularly monitor fairness metrics during model evaluation.
5. Include human oversight in hiring decisions.

## Conclusion
A model should not only be accurate but also fair. Bias audits are essential to ensure machine learning systems make ethical and equitable decisions.